# TabPFN → drzewo decyzyjne (rozwiązanie 2)

Colab: **Runtime → Change runtime type → T4 GPU**, potem Run all.

Pipeline destylacji:

1. **TabPFN** trenuje się wyłącznie na dozwolonym `val.csv` (holdout i test nie wchodzą).
2. Nauczyciel **dopisuje etykiety** do nieoznaczonego `train.csv` (pseudo-labelki).
3. **Student (drzewo)** uczy się ze znacznie większego zbioru: `val` ∪ `train`.

W notebooku są **dwa** drzewa na tym samym nauczycielu, żeby porównać z wariantem „tylko val”. Aplikacja (`app_tabpfn.py`) dostaje drzewo z powiększonego zbioru — CPU, ścieżka if/then.

**Weryfikacja:** uczciwy Raw_Score na `final_valid.csv` z wstrzykniętymi lukami pomiarowymi (~5%, jak `test.csv`). Submit zostaje na prawdziwie dziurawym `test.csv`.

In [8]:
import sys
from pathlib import Path


IN_COLAB = "google.colab" in sys.modules
IN_COLAB = False
if IN_COLAB:
    %pip install -q tabpfn scikit-learn pandas numpy torch joblib plotly
    from google.colab import files
    print("Wgraj val.csv, final_valid.csv, train.csv, test.csv oraz tabpfn_diagnose.py")
    files.upload()

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

cuda: False CPU


In [9]:
from tabpfn_diagnose import (
    TabPFNTreeDiagnoser,
    hackathon_score,
    pick_device,
    print_eval,
    punch_spectrum_gaps,
    read_labeled_csv,
)
import pandas as pd

val = read_labeled_csv("val.csv")
holdout = read_labeled_csv("final_valid.csv")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
overlap = set(val["engine_id"]) & set(holdout["engine_id"])
assert not overlap, f"wyciek silników val ∩ final_valid: {sorted(overlap)}"
print(
    "val", val.shape, "final_valid", holdout.shape,
    "train", train.shape, "test", test.shape,
    "device=", pick_device(),
)
print("silniki holdout:", sorted(holdout["engine_id"].unique()))
print("symulacja luk warsztatowych na val / final_valid (train/test już je mają):")
val = punch_spectrum_gaps(val, verbose=True)
holdout = punch_spectrum_gaps(holdout, verbose=True)
punch_spectrum_gaps(train, verbose=True)
punch_spectrum_gaps(test, verbose=True)

val (328, 26) final_valid (148, 26) train (2400, 24) test (600, 24) device= cpu
silniki holdout: ['val_0000', 'val_0001', 'val_0002', 'val_0003', 'val_0004', 'val_0005', 'val_0006', 'val_0007', 'val_0008', 'val_0009', 'val_0010', 'val_0011']


## Nauczyciel: TabPFN na `val.csv`

TabPFN widzi wyłącznie dozwolony valid (z wstrzykniętymi lukami jak w `test.csv`). Najpierw destylujemy drzewo **tylko z val** — holdout (też z lukami) zapisujemy do porównania. Na T4 możesz zostawić `do_cv=True` (GroupKFold po silniku). Na CPU: `do_cv=False`.

In [10]:
device = pick_device()
n_estimators = 8 if device == "cuda" else 4
do_cv = device == "cuda"  # T4: extra GroupKFold na val; CPU: pomiń
CONF_MIN = 0.70  # pewność pseudo-etykiet; 0.0 = weź cały train.csv

model = TabPFNTreeDiagnoser().fit(
    val,
    train=None,  # student A: tylko val
    device=device,
    n_estimators=n_estimators,
    do_cv=do_cv,
)
sub_ho_tree_val_only = model.predict(holdout)
print("student A (tylko val)  n_student=", model.meta.get("n_student"))
print(model.meta)

TabPFN teacher  device=cpu  n_estimators=4
TabPFN in-sample on val.csv (sanity, not the holdout score):
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       279
 zakoksowany      1.000     1.000     1.000        12
      lejacy      1.000     1.000     1.000        10
       pompa      1.000     1.000     1.000         6
      iglica      1.000     1.000     1.000        13
     unknown      1.000     1.000     1.000         8

    accuracy                          1.000       328
   macro avg      1.000     1.000     1.000       328
weighted avg      1.000     1.000     1.000       328

Student tree distilled on 328 rows (val=328, pseudo=0)

Distilled tree on val.csv (resubstitution, not holdout):
              precision    recall  f1-score   support

          ok      1.000     0.996     0.998       279
 zakoksowany      1.000     1.000     1.000        12
      lejacy      1.000     1.000     1.000        10
       pompa      0.667

## Student z powiększonym zbiorem: `val` ∪ `train`

TabPFN (już wytrenowany) etykietuje `train.csv`. Drzewo destylujemy ze **znacznie większego** zbioru: prawdziwe labelki z val + pseudo-labelki z train. Nauczyciel się nie trenuje drugi raz.

Porównanie na `final_valid.csv` z lukami: teacher / drzewo←val / drzewo←val+train. Do aplikacji zapisujemy wariant z train.

In [11]:
y_ho = holdout["label"].to_numpy()
s_ho = holdout["severity"].to_numpy()
sub_ho_tabpfn = model.predict_teacher(holdout, model.teacher_)
n_fit_val = int(model.meta["n_student"])

model.distill(train, conf_min=CONF_MIN)  # student B: val + pseudo-train
sub_ho_tree = model.predict(holdout)
n_fit_big = int(model.meta["n_student"])
model.save()

rows = []
for name, sub, n_fit in [
    ("TabPFN teacher", sub_ho_tabpfn, len(val)),
    ("drzewo ← tylko val", sub_ho_tree_val_only, n_fit_val),
    ("drzewo ← val + pseudo-train", sub_ho_tree, n_fit_big),
]:
    raw, macro, sev = hackathon_score(
        y_ho, sub["label"].to_numpy(), s_ho, sub["severity"].to_numpy()
    )
    agree = float((sub["label"].to_numpy() == sub_ho_tabpfn["label"].to_numpy()).mean())
    rows.append(
        {
            "model": name,
            "n_fit": n_fit,
            "Raw_Score": raw,
            "macro-F1": macro,
            "severity_acc": sev,
            "zgoda vs TabPFN": agree,
        }
    )
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(
    f"\npseudo-labelki: {model.meta.get('n_pseudo')}/{len(train)} "
    f"(próg P ≥ {CONF_MIN:.2f})  →  student {n_fit_val} → {n_fit_big} wierszy"
)
print("rozkład pseudo-etykiet:", model.meta.get("train_pseudo", {}).get("label_counts"))

Pseudo-labels from train.csv: 2367/2400 with max P ≥ 0.70
ok             2033
unknown          78
zakoksowany      67
iglica           66
pompa            62
lejacy           61
Student tree distilled on 2695 rows (val=328, pseudo=2367)

Distilled tree on val.csv (resubstitution, not holdout):
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       279
 zakoksowany      1.000     1.000     1.000        12
      lejacy      1.000     1.000     1.000        10
       pompa      1.000     1.000     1.000         6
      iglica      1.000     1.000     1.000        13
     unknown      1.000     1.000     1.000         8

    accuracy                          1.000       328
   macro avg      1.000     1.000     1.000       328
weighted avg      1.000     1.000     1.000       328

tree vs val labels  macro-F1=1.0000  severity_acc=1.0000  Raw_Score=1.0000  fidelity vs TabPFN=1.000
Wrote /home/janek/Desktop/hackathon-engin/artifacts/diagnoser

## Holdout: `final_valid.csv` — raporty klas

Silniki spoza `val.csv`. Poniżej pełny classification_report dla nauczyciela i **wybranego** studenta (val + pseudo-train). Tabelka porównawcza jest w komórce wyżej.

In [12]:
print_eval("final_valid — TabPFN teacher", y_ho, sub_ho_tabpfn["label"].to_numpy(), s_ho, sub_ho_tabpfn["severity"].to_numpy())
print_eval("final_valid — drzewo ← val + pseudo-train", y_ho, sub_ho_tree["label"].to_numpy(), s_ho, sub_ho_tree["severity"].to_numpy())
print_eval("final_valid — drzewo ← tylko val", y_ho, sub_ho_tree_val_only["label"].to_numpy(), s_ho, sub_ho_tree_val_only["severity"].to_numpy())
print(
    f"zgoda drzewo(val+train) vs TabPFN: {(sub_ho_tree['label'] == sub_ho_tabpfn['label']).mean():.3f}  |  "
    f"zgoda drzewo(tylko val) vs TabPFN: {(sub_ho_tree_val_only['label'] == sub_ho_tabpfn['label']).mean():.3f}"
)


=== final_valid — TabPFN teacher ===


              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       128
 zakoksowany      1.000     1.000     1.000         6
      lejacy      1.000     1.000     1.000         4
       pompa      1.000     1.000     1.000         3
      iglica      1.000     1.000     1.000         3
     unknown      1.000     1.000     1.000         4

    accuracy                          1.000       148
   macro avg      1.000     1.000     1.000       148
weighted avg      1.000     1.000     1.000       148

macro-F1=1.0000  severity_acc=1.0000  Raw_Score=1.0000

=== final_valid — drzewo ← val + pseudo-train ===
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       128
 zakoksowany      1.000     1.000     1.000         6
      lejacy      1.000     1.000     1.000         4
       pompa      1.000     1.000     1.000         3
      iglica      1.000     1.000     1.000         3
     unknown      1.000   

## Drzewo, które zobaczy mechanik

In [13]:
print(model.rules_text())

|--- residual 9 kHz vs. baseline silnika <= -8.43
|   |--- dołek przy 9 kHz (koks) <= -6.45
|   |   |--- class: zakoksowany
|   |--- dołek przy 9 kHz (koks) >  -6.45
|   |   |--- podobieństwo do wzorca: zakoksowany <= -0.30
|   |   |   |--- residual 15 kHz vs. baseline silnika <= -15.38
|   |   |   |   |--- class: lejacy
|   |   |   |--- residual 15 kHz vs. baseline silnika >  -15.38
|   |   |   |   |--- class: pompa
|   |   |--- podobieństwo do wzorca: zakoksowany >  -0.30
|   |   |   |--- podobieństwo do wzorca: iglica <= 0.94
|   |   |   |   |--- podobieństwo do wzorca: pompa <= 0.87
|   |   |   |   |   |--- odchyłka L2 od profilu silnika <= 39.80
|   |   |   |   |   |   |--- class: ok
|   |   |   |   |   |--- odchyłka L2 od profilu silnika >  39.80
|   |   |   |   |   |   |--- class: unknown
|   |   |   |   |--- podobieństwo do wzorca: pompa >  0.87
|   |   |   |   |   |--- residual 16 kHz vs. baseline silnika <= -2.02
|   |   |   |   |   |   |--- class: pompa
|   |   |   |   |   |

## Submit `test.csv` + zgodność nauczyciel / student

Ten sam nauczyciel (fit na `val.csv`) i student z `val` ∪ `train`. `test.csv` nie ma etykiet.

In [14]:
sub_tree = model.predict(test)
sub_tabpfn = model.predict_teacher(test, model.teacher_)
sub_tree.to_csv("predictions_tree.csv", index=False)
sub_tabpfn.to_csv("predictions_tabpfn.csv", index=False)
agree = (sub_tree["label"] == sub_tabpfn["label"]).mean()
print(f"zgoda drzewo vs TabPFN na teście (bez etykiet): {agree:.3f}")
print("TabPFN\n", sub_tabpfn["label"].value_counts())
print("drzewo\n", sub_tree["label"].value_counts())

if IN_COLAB:
    files.download("predictions_tabpfn.csv")
    files.download("predictions_tree.csv")
    files.download("artifacts/diagnoser_tree.joblib")

zgoda drzewo vs TabPFN na teście (bez etykiet): 0.992
TabPFN
 label
ok             518
unknown         21
zakoksowany     16
pompa           16
iglica          16
lejacy          13
Name: count, dtype: int64
drzewo
 label
ok             517
unknown         25
zakoksowany     16
pompa           16
lejacy          13
iglica          13
Name: count, dtype: int64


Lokalnie po pobraniu `diagnoser_tree.joblib` do `artifacts/`:

```bash
streamlit run app_tabpfn.py
```

## (opcjonalnie) LOEO na `val ∪ final_valid`

Plików CSV **nie ruszamy**. W RAM sklejamy etykietowane zbiory, wstrzykujemy luki jak w `test.csv` i robimy leave-one-engine-out dla trzech modeli z tego notebooka:

1. TabPFN teacher
2. drzewo destylowane z foldu + pseudo-`train.csv`
3. drzewo destylowane tylko z etykiet foldu

Na foldzie: szablony, nauczyciel i progi severity nie widzą held-out silnika. Na CPU to jest wolne (refit TabPFN × liczba silników). Ustaw `RUN_LOEO = True` i odpal komórkę.


In [ ]:
from tabpfn_diagnose import leave_one_engine_out, pick_device, read_labeled_csv
import pandas as pd

RUN_LOEO = True  # True → concat val ∪ final_valid w RAM, LOEO po silniku

if not RUN_LOEO:
    print("pominięte — ustaw RUN_LOEO = True")
else:
    val_loeo = read_labeled_csv("val.csv")
    ho_loeo = read_labeled_csv("final_valid.csv")
    train_loeo = pd.read_csv("train.csv")
    device_loeo = pick_device()
    n_est = 8 if device_loeo == "cuda" else 4
    labeled = pd.concat([val_loeo, ho_loeo], ignore_index=True)
    print("LOEO na val ∪ final_valid  (concat tylko w RAM, plików nie zapisujemy)")
    print(f"  silniki: {labeled['engine_id'].nunique()}  cylindry: {len(labeled)}")
    print(f"  z val: {len(val_loeo)}  z final_valid: {len(ho_loeo)}  device={device_loeo}")
    leave_one_engine_out(
        labeled,
        train_loeo,
        device=device_loeo,
        n_estimators=n_est,
        conf_min=0.70,
    )
